# 面试问题：怎样评测 LLM 的长上下文“Lost in the Middle”位置鲁棒性？

**一句话回答。** 不能只测一个超长 prompt：应控制总长度、needle 的相对位置、干扰类型、问题模板、答案解析和无答案/冲突对照，按位置×长度切片报告准确率与置信区间；合成 needle 成功不等于真实长文理解。

本题只用 Python 标准库重建数据合同、评分、状态机和失败分支；断言只验证小型受控样例，不能替代真实模型质量、长上下文能力或线上容量压测。

**资料入口。** [Lost in the Middle](https://arxiv.org/abs/2307.03172) 发现相关信息位于上下文中部时性能可能明显下降；本题实现的是评测 harness，而不是伪造模型能力。


In [ ]:
question = "long-context position robustness"  # 执行本行的状态、计算或校验逻辑。
assert "position" in question  # 执行本行的状态、计算或校验逻辑。
assert 5 >= 5  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 测试合同固定模型、模板和切片坐标

相对位置必须按 token 计而非字符计；教学用列表模拟 token。每个 case 记录 context length、needle index、query、期望答案、模型/tokenizer/template revision，避免把提示变化误判成位置效应。


In [ ]:
config = {"model": "m1", "tokenizer": "t1", "template": "v1", "seed": 7}  # 执行本行的状态、计算或校验逻辑。
positions = {"front": 0, "middle": 5, "back": 9}  # 执行本行的状态、计算或校验逻辑。
assert set(positions) == {"front", "middle", "back"}  # 执行本行的状态、计算或校验逻辑。
assert config["seed"] == 7  # 执行本行的状态、计算或校验逻辑。
assert positions["middle"] < positions["back"]  # 执行本行的状态、计算或校验逻辑。

## 2. 构造 needle 与干扰项，同时保留坐标

干扰文本不能意外包含答案，也不能在不同位置改变长度分布。真实评测还应使用自然文档、多事实、相似干扰和任务标签；这里先验证生成 case 的位置和长度合同。


In [ ]:
def make_case(length, index, needle):  # 执行本行的状态、计算或校验逻辑。
    tokens = ["distractor" for _ in range(length)]  # 执行本行的状态、计算或校验逻辑。
    tokens[index] = needle  # 执行本行的状态、计算或校验逻辑。
    return {"tokens": tokens, "needle_index": index, "needle": needle, "length": length}  # 执行本行的状态、计算或校验逻辑。
front_case = make_case(10, positions["front"], "KEY=42")  # 执行本行的状态、计算或校验逻辑。
middle_case = make_case(10, positions["middle"], "KEY=42")  # 执行本行的状态、计算或校验逻辑。
assert front_case["tokens"][0] == "KEY=42"  # 执行本行的状态、计算或校验逻辑。
assert middle_case["tokens"][5] == "KEY=42"  # 执行本行的状态、计算或校验逻辑。
assert len(middle_case["tokens"]) == 10  # 执行本行的状态、计算或校验逻辑。

## 3. 独立验证数据本身，避免题目泄漏

先检查 needle 唯一、问题不含答案、上下文总长度正确；否则模型可能凭题面猜中，或多处同值使“位置”没有定义。无答案 case 与冲突 case 是必须的负对照。


In [ ]:
def valid_case(case, query):  # 执行本行的状态、计算或校验逻辑。
    return case["tokens"].count(case["needle"]) == 1 and case["needle"] not in query and len(case["tokens"]) == case["length"]  # 执行本行的状态、计算或校验逻辑。
assert valid_case(front_case, "请返回 KEY 的值")  # 执行本行的状态、计算或校验逻辑。
assert valid_case(middle_case, "请返回 KEY 的值")  # 执行本行的状态、计算或校验逻辑。
assert not valid_case(front_case, "KEY=42 是什么")  # 执行本行的状态、计算或校验逻辑。

## 4. 答案 parser 与评分先于模型输出定义

模型可能回答 `42`、`KEY=42` 或包含解释。评测应采用任务特定、确定性的 parser 并记录 parse failure；不能让 LLM judge 在位置实验中引入另一层不可控位置/风格偏差。


In [ ]:
def parse_key(answer):  # 执行本行的状态、计算或校验逻辑。
    return "42" if "42" in answer else None  # 执行本行的状态、计算或校验逻辑。
assert parse_key("答案是 42") == "42"  # 执行本行的状态、计算或校验逻辑。
assert parse_key("KEY=42，因为命中") == "42"  # 执行本行的状态、计算或校验逻辑。
assert parse_key("我不知道") is None  # 执行本行的状态、计算或校验逻辑。

## 5. 按位置运行同一问题，不把合成输出当真实性能

这里的 outputs 是受控测试替身：中部失败用于检查 harness 能否分片统计，并非任何真实模型的分数。真实运行需要固定服务快照、重复采样并保存 prompt/token counts。


In [ ]:
outputs = {"front": "42", "middle": "我不知道", "back": "42"}  # 执行本行的状态、计算或校验逻辑。
def correct(output, gold):  # 执行本行的状态、计算或校验逻辑。
    return int(parse_key(output) == gold)  # 执行本行的状态、计算或校验逻辑。
position_scores = {name: correct(output, "42") for name, output in outputs.items()}  # 执行本行的状态、计算或校验逻辑。
assert position_scores == {"front": 1, "middle": 0, "back": 1}  # 执行本行的状态、计算或校验逻辑。
assert sum(position_scores.values()) == 2  # 执行本行的状态、计算或校验逻辑。
assert len(outputs) == 3  # 执行本行的状态、计算或校验逻辑。

## 6. 长度×位置是二维表，而不是一个平均数

中部失败可能只出现在更长长度，或源于 prompt 截断。将长度 bin 与位置 bin 交叉汇总，再输出每个 cell 的样本数；没有样本的 cell 不能被均值悄悄填零。


In [ ]:
runs = [{"length": 10, "position": "front", "score": 1}, {"length": 10, "position": "middle", "score": 0}, {"length": 20, "position": "front", "score": 1}, {"length": 20, "position": "middle", "score": 0}]  # 执行本行的状态、计算或校验逻辑。
def mean_score(rows):  # 执行本行的状态、计算或校验逻辑。
    return sum(row["score"] for row in rows) / len(rows) if rows else None  # 执行本行的状态、计算或校验逻辑。
middle_rows = [row for row in runs if row["position"] == "middle"]  # 执行本行的状态、计算或校验逻辑。
assert mean_score(middle_rows) == 0.0  # 执行本行的状态、计算或校验逻辑。
assert mean_score([]) is None  # 执行本行的状态、计算或校验逻辑。
assert len({row["length"] for row in runs}) == 2  # 执行本行的状态、计算或校验逻辑。

## 7. 无答案与冲突对照检验“猜中”

如果没有 needle，模型应拒答而非从训练记忆猜一个值；若存在两个冲突 needle，应按预先定义的规则拒答或指出冲突。否则 front/back 高分可能只是模式补全。


In [ ]:
absent = make_case(10, 0, "distractor")  # 执行本行的状态、计算或校验逻辑。
conflict = ["KEY=42", "distractor", "KEY=99"]  # 执行本行的状态、计算或校验逻辑。
assert "KEY=42" not in absent["tokens"]  # 执行本行的状态、计算或校验逻辑。
assert len([item for item in conflict if item.startswith("KEY=")]) == 2  # 执行本行的状态、计算或校验逻辑。
assert parse_key("无法确定") is None  # 执行本行的状态、计算或校验逻辑。

## 8. 发布结果时绑定配置与限制

报告应包含位置曲线、长度曲线、样本数、解析失败、无答案 false positive、延迟/成本和模型版本。仅凭该合成 harness 不能说明真实文档 QA、推理或安全能力，必须与自然任务和回归集并列。


In [ ]:
def comparable(run_config, current_config):  # 执行本行的状态、计算或校验逻辑。
    return all(run_config[key] == current_config[key] for key in ("model", "tokenizer", "template"))  # 执行本行的状态、计算或校验逻辑。
assert comparable(config, {**config, "seed": 99})  # 执行本行的状态、计算或校验逻辑。
assert not comparable(config, {**config, "model": "m2"})  # 执行本行的状态、计算或校验逻辑。
assert all(value in (0, 1) for value in position_scores.values())  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试答案要把“模型宣称支持 N token”与“在每个位置可靠使用 N token”分开。先建立 token 坐标和测试控制，再评测位置×长度×干扰、负对照和 parser，最后用自然数据与线上任务验证；不能用一个 needle 结果推导真实长文理解。
